# Bloco 4 — Protótipo do classificador (4.1 a 4.3)

Protótipo funcional da automação proposta no Bloco 3. Um classificador linear
sobre o D2 com métrica real em holdout, mais a camada de abstenção que desvia o
ticket de baixa confiança para revisão humana.

| Seção | Pergunta | Entrega |
|-------|----------|---------|
| 4.1 | Quanto sinal o D2 permite recuperar com um modelo simples? | Classificador base, F1-macro em holdout contra baseline |
| 4.2 | Desviar o ticket incerto melhora a fração automatizada, e o que cai na revisão? | Camada de abstenção, trade-off cobertura contra qualidade |
| 4.3 | Como isso vira um contrato de inferência? | Função de decisão, exemplo e síntese |

Métrica principal é F1-macro. As oito classes do D2 são desbalanceadas e a
acurácia premia a maioria. O F1-macro dá peso igual a cada classe. Todo número
abaixo vem de holdout estratificado, com o vetorizador ajustado apenas no treino.

## 4.0 — Setup e carga

Parâmetros, imports e recarga autossuficiente do D2. O notebook roda do zero por
Restart & Run All. Nenhum estado é importado dos notebooks anteriores.

In [1]:
# Parametros do Bloco 4. Ajuste aqui.
TEST_SIZE = 0.2         # fracao do holdout
RANDOM_STATE = 42       # reprodutibilidade do split e do modelo

MIN_DF = 5              # termo entra no vocabulario se aparece em >= 5 documentos de treino
NGRAM_MAX = 2           # TF-IDF de unigramas e bigramas
THR_GRID = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]   # grade de limiares de confianca
THR_OP = 0.5            # ponto de operacao da abstencao, justificado em 4.2

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
def load_dataset(filename: str) -> pd.DataFrame:
    """Carrega um CSV de `solution/datasets/` e retorna o DataFrame.

    Mesma funcao de carga dos notebooks anteriores. O caminho e resolvido de
    forma relativa: procura `datasets/` a partir do diretorio de execucao
    (`solution/`) e, como fallback, sobe na arvore ate achar `solution/datasets/`.
    Sem nenhum caminho absoluto de maquina.
    """
    candidates = [Path("datasets")]
    candidates += [parent / "solution" / "datasets" for parent in [Path.cwd(), *Path.cwd().parents]]
    data_dir = next((d for d in candidates if d.is_dir()), None)
    if data_dir is None:
        raise FileNotFoundError("Pasta solution/datasets/ nao encontrada a partir de " + str(Path.cwd()))
    return pd.read_csv(data_dir / filename, encoding="utf-8", low_memory=False)


df2 = load_dataset("all_tickets_processed_improved_v3.csv")
X = df2["Document"].astype(str)
y = df2["Topic_group"]
print("D2:", df2.shape[0], "tickets,", y.nunique(), "classes")
display(y.value_counts().rename("tickets").to_frame())

D2: 47837 tickets, 8 classes


,tickets
Topic_group,
Hardware,13617
HR Support,10915
Access,7125
Miscellaneous,7060
Storage,2777
Purchase,2464
Internal Project,2119
Administrative rights,1760


## 4.1 — Classificador base com holdout

Pergunta: quanto sinal real o D2 permite recuperar com um modelo simples?

Split estratificado por `Topic_group`, TF-IDF ajustado só no treino e uma
regressão logística linear. O modelo devolve probabilidade por classe, o que
alimenta a camada de abstenção de 4.2. `class_weight="balanced"` compensa o
desbalanceamento das classes, coerente com a escolha de otimizar F1-macro.

In [3]:
# Split estratificado: mantem a proporcao das 8 classes no treino e no holdout.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
y_te = y_te.reset_index(drop=True)
print("treino:", X_tr.shape[0], " holdout:", X_te.shape[0])

# Proporcao por classe preservada no holdout.
prop = pd.DataFrame({
    "treino": y_tr.value_counts(normalize=True),
    "holdout": y_te.value_counts(normalize=True),
}).round(4)
display(prop)

treino: 38269  holdout: 9568


,treino,holdout
Topic_group,,
Hardware,0.2846,0.2847
HR Support,0.2282,0.2282
Access,0.1489,0.1489
Miscellaneous,0.1476,0.1476
Storage,0.0581,0.0580
Purchase,0.0515,0.0515
Internal Project,0.0443,0.0443
Administrative rights,0.0368,0.0368


In [4]:
# TF-IDF ajustado apenas no treino, depois aplicado ao holdout.
# Ajustar no conjunto completo vazaria vocabulario do holdout para o treino.
vectorizer = TfidfVectorizer(min_df=MIN_DF, ngram_range=(1, NGRAM_MAX), sublinear_tf=True)
X_tr_vec = vectorizer.fit_transform(X_tr)
X_te_vec = vectorizer.transform(X_te)
print("vocabulario:", len(vectorizer.vocabulary_), "termos")

vocabulario: 45427 termos


In [5]:
# Classificador linear e inspecionavel sobre TF-IDF.
clf = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=RANDOM_STATE)
clf.fit(X_tr_vec, y_tr)
pred = pd.Series(clf.predict(X_te_vec), index=y_te.index)

In [6]:
# F1-macro do modelo contra o baseline de classe majoritaria.
majority = y_tr.value_counts().idxmax()
baseline_pred = pd.Series([majority] * len(y_te), index=y_te.index)

f1_base = f1_score(y_te, pred, average="macro")
f1_baseline = f1_score(y_te, baseline_pred, average="macro")

display(Markdown(
    f"| Modelo | F1-macro no holdout |\n|---|---|\n"
    f"| Regressao logistica sobre TF-IDF | **{f1_base:.4f}** |\n"
    f"| Baseline (sempre `{majority}`) | {f1_baseline:.4f} |"
))

| Modelo | F1-macro no holdout |
|---|---|
| Regressao logistica sobre TF-IDF | **0.8560** |
| Baseline (sempre `Hardware`) | 0.0554 |

In [7]:
# Relatorio por classe: precision, recall e F1 por Topic_group no holdout.
report = pd.DataFrame(classification_report(y_te, pred, output_dict=True)).T
per_class = report.drop(index=["accuracy", "macro avg", "weighted avg"])
per_class = per_class.rename(columns={"f1-score": "f1"})[["precision", "recall", "f1", "support"]]
per_class["support"] = per_class["support"].astype(int)
display(per_class.round(3).sort_values("f1"))

,precision,recall,f1,support
Administrative rights,0.690,0.855,0.764,352
Miscellaneous,0.797,0.864,0.829,1412
Hardware,0.864,0.815,0.839,2724
Internal Project,0.800,0.925,0.858,424
HR Support,0.895,0.846,0.870,2183
Storage,0.858,0.915,0.886,555
Access,0.896,0.886,0.891,1425
Purchase,0.925,0.899,0.912,493


In [8]:
# Matriz de confusao no holdout como tabela. Linha = verdadeiro, coluna = predito.
labels = sorted(y.unique())
cm = confusion_matrix(y_te, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = "verdadeiro \\ predito"
display(cm_df)

,Access,Administrative rights,HR Support,Hardware,Internal Project,Miscellaneous,Purchase,Storage
verdadeiro \ predito,,,,,,,,
Access,1263,20,22,52,16,41,2,9
Administrative rights,4,301,4,36,1,6,0,0
HR Support,41,12,1846,126,34,93,6,25
Hardware,80,84,120,2220,25,142,26,27
Internal Project,1,1,7,13,392,9,0,1
Miscellaneous,17,11,46,76,20,1220,2,20
Purchase,2,3,7,26,1,9,443,2
Storage,1,4,10,21,1,10,0,508


### Leitura 4.1

O modelo entrega F1-macro 0.856 no holdout, contra 0.055 do baseline de classe
majoritária. O ganho é de mais de quinze vezes. O baseline colapsa porque prevê
sempre `Hardware` e zera o F1 das outras sete classes. Um modelo linear simples
sobre o texto do D2 recupera sinal real e separável nas oito classes.

O modelo não abandona nenhuma classe. O F1 por classe fica entre 0.76 e 0.91,
sem nenhuma classe perto de zero. As classes com vocabulário próprio são as mais
fáceis: `Purchase` (F1 0.912), `Access` (0.891) e `Storage` (0.886). As mais
difíceis são `Administrative rights` (0.764), onde a precision cai para 0.69, e
`Miscellaneous` (0.829), a categoria difusa diagnosticada no Bloco 2. A matriz
de confusão mostra que os erros se concentram em `Hardware`, que recebe
predições indevidas de quase todas as outras classes. `Hardware` é ao mesmo
tempo a maior classe e o principal sorvedouro de confusão do modelo.

## 4.2 — Camada de abstenção

Pergunta: desviar o ticket de baixa confiança melhora a fração automatizada, e o
que cai na revisão humana?

O score de confiança é a probabilidade máxima da regressão logística. Um limiar
sobre esse score separa o ticket roteado automaticamente do ticket desviado para
revisão humana.

In [9]:
# Score de confianca por predicao: a maior probabilidade entre as 8 classes.
proba = clf.predict_proba(X_te_vec)
confidence = pd.Series(proba.max(axis=1), index=y_te.index)
print("confianca no holdout: min %.3f  mediana %.3f  max %.3f"
      % (confidence.min(), confidence.median(), confidence.max()))

confianca no holdout: min 0.190  mediana 0.759  max 1.000


In [10]:
# Trade-off cobertura contra qualidade ao longo da grade de limiares.
# Para cada limiar: fracao automatizada e F1-macro dessa fracao.
rows = []
for thr in THR_GRID:
    auto = confidence >= thr
    cov = auto.mean()
    f1_auto = f1_score(y_te[auto], pred[auto], average="macro") if auto.sum() else np.nan
    rows.append({"limiar": thr, "cobertura": round(cov, 3), "f1_macro_auto": round(f1_auto, 4)})
tradeoff = pd.DataFrame(rows)
display(tradeoff)

,limiar,cobertura,f1_macro_auto
0,0.0,1.000,0.8560
1,0.3,0.970,0.8694
2,0.4,0.881,0.9016
3,0.5,0.767,0.9308
4,0.6,0.660,0.9561
5,0.7,0.559,0.9725
6,0.8,0.455,0.9830
7,0.9,0.325,0.9902


In [11]:
# Ponto de operacao escolhido: THR_OP.
# Justificativa: em THR_OP=0.5 a fracao automatizada sobe de F1-macro 0.856 para
# ~0.93 mantendo cobertura acima de 3/4 dos tickets. Limiares maiores seguem
# melhorando a qualidade, mas derrubam a cobertura rapido (0.66 em 0.6, 0.46 em
# 0.8). 0.5 e o limiar natural de maioria probabilistica e fica no joelho da curva.
auto_mask = confidence >= THR_OP
coverage = auto_mask.mean()
f1_auto = f1_score(y_te[auto_mask], pred[auto_mask], average="macro")

display(Markdown(
    f"**Ponto de operacao: limiar {THR_OP}**\n\n"
    f"| Fracao | Cobertura | F1-macro |\n|---|---|---|\n"
    f"| Geral (4.1, sem abstencao) | 1.000 | {f1_base:.4f} |\n"
    f"| Automatizada (confianca >= {THR_OP}) | {coverage:.3f} | **{f1_auto:.4f}** |\n"
    f"| Desviada para revisao humana | {1 - coverage:.3f} | — |"
))

**Ponto de operacao: limiar 0.5**

| Fracao | Cobertura | F1-macro |
|---|---|---|
| Geral (4.1, sem abstencao) | 1.000 | 0.8560 |
| Automatizada (confianca >= 0.5) | 0.767 | **0.9308** |
| Desviada para revisao humana | 0.233 | — |

In [12]:
# Composicao por Topic_group da fracao desviada, comparada com a populacao.
# ratio > 1: classe sobre-representada no desvio. ratio < 1: sub-representada.
deviated_mask = ~auto_mask
pop_share = y_te.value_counts(normalize=True)
dev_share = y_te[deviated_mask].value_counts(normalize=True)
composition = pd.DataFrame({"pop": pop_share, "desvio": dev_share}).fillna(0.0)
composition["ratio"] = (composition["desvio"] / composition["pop"]).round(3)
composition = composition.round(3).sort_values("ratio", ascending=False)
display(composition)

over = composition.index[composition["ratio"] > 1].tolist()
under = composition.index[composition["ratio"] < 1].tolist()
print("sobre-representadas no desvio:", ", ".join(over))
print("sub-representadas no desvio:", ", ".join(under))

,pop,desvio,ratio
Topic_group,,,
Hardware,0.285,0.388,1.362
Miscellaneous,0.148,0.157,1.065
HR Support,0.228,0.240,1.050
Administrative rights,0.037,0.034,0.913
Access,0.149,0.102,0.683
Internal Project,0.044,0.027,0.606
Storage,0.058,0.031,0.540
Purchase,0.052,0.022,0.426


sobre-representadas no desvio: Hardware, Miscellaneous, HR Support
sub-representadas no desvio: Administrative rights, Access, Internal Project, Storage, Purchase


### Leitura 4.2

A abstenção funciona. No ponto de operação, limiar 0.5, a fração automatizada
sobe de F1-macro 0.856 para 0.931, cobrindo 76.7% dos tickets. Os 23.3%
restantes vão para revisão humana. A automação passa a operar sobre o
subconjunto onde o modelo é confiável, e a qualidade da fração automatizada sobe
quase oito pontos de F1-macro em troca de um quarto do volume.

O perfil do desvio distingue vocabulário difuso de dificuldade de classificação.
A classe mais sobre-representada na revisão humana é `Hardware`, com ratio 1.36,
a maior classe do D2. `Miscellaneous`, a categoria difusa no vocabulário
diagnosticada no Bloco 2, aparece só levemente sobre-representada (1.07), junto
de `HR Support` (1.05). `Purchase` (0.43), `Storage` (0.54) e `Internal Project`
(0.61) ficam sub-representadas, automatizadas quase por inteiro. O modelo rotula
o `Miscellaneous` com F1 0.829, então o vocabulário difuso não trava a
classificação. A incerteza se concentra em `Hardware`, o sorvedouro de confusão
de 4.1, e a revisão humana herda sobretudo o ticket de `Hardware` ambíguo.

## 4.3 — Contrato de inferência e síntese

Uma função que recebe um texto e devolve categoria, confiança e a decisão entre
roteamento automático e revisão humana, no ponto de operação de 4.2. Em seguida,
um exemplo do holdout e o dict `resultado` com os números-chave do bloco.

In [13]:
def classificar_ticket(texto: str) -> dict:
    """Recebe o texto de um ticket e devolve categoria, confianca e decisao.

    Usa o vetorizador e o classificador ajustados acima e o ponto de operacao
    THR_OP. A decisao e "roteamento automatico" quando a confianca alcanca o
    limiar, "revisao humana" caso contrario.
    """
    vec = vectorizer.transform([str(texto)])
    probs = clf.predict_proba(vec)[0]
    idx = int(probs.argmax())
    categoria = clf.classes_[idx]
    conf = float(probs[idx])
    decisao = "roteamento automatico" if conf >= THR_OP else "revisao humana"
    return {"categoria": categoria, "confianca": round(conf, 4), "decisao": decisao}

In [14]:
# Exemplo inspecionavel: um ticket do holdout, com texto, predicao, confianca e decisao.
ex_pos = y_te.index[0]
ex_texto = X_te.iloc[0]
ex_out = classificar_ticket(ex_texto)

display(Markdown(
    f"**Exemplo (ticket do holdout)**\n\n"
    f"- Texto (inicio): `{ex_texto[:180]}...`\n"
    f"- Rotulo verdadeiro: `{y_te.loc[ex_pos]}`\n"
    f"- Categoria predita: `{ex_out['categoria']}`\n"
    f"- Confianca: {ex_out['confianca']}\n"
    f"- Decisao: **{ex_out['decisao']}**"
))

**Exemplo (ticket do holdout)**

- Texto (inicio): `project codes july pm codes hello please assign task thank kind regards july pm codes hi pm attached codes pm thanks...`
- Rotulo verdadeiro: `Internal Project`
- Categoria predita: `Internal Project`
- Confianca: 0.9919
- Decisao: **roteamento automatico**

In [15]:
# Dict resultado, preenchido a partir dos valores computados no bloco.
over_txt = ", ".join(composition.index[composition["ratio"] > 1].tolist())
under_txt = ", ".join(composition.index[composition["ratio"] < 1].tolist())

resultado = {
    "f1_macro_base": round(float(f1_base), 4),
    "f1_macro_baseline": round(float(f1_baseline), 4),
    "f1_macro_automatica": round(float(f1_auto), 4),
    "cobertura_ponto_operacao": round(float(coverage), 4),
    "limiar_operacao": THR_OP,
    "composicao_desvio_sobre": over_txt,
    "composicao_desvio_sub": under_txt,
}
resultado

{'f1_macro_base': 0.856,
 'f1_macro_baseline': 0.0554,
 'f1_macro_automatica': 0.9308,
 'cobertura_ponto_operacao': 0.7666,
 'limiar_operacao': 0.5,
 'composicao_desvio_sobre': 'Hardware, Miscellaneous, HR Support',
 'composicao_desvio_sub': 'Administrative rights, Access, Internal Project, Storage, Purchase'}

In [16]:
# Renderizacao do dict resultado como tabela markdown.
linhas = "\n".join(f"| `{k}` | {v} |" for k, v in resultado.items())
display(Markdown("| Chave | Valor |\n|---|---|\n" + linhas))

| Chave | Valor |
|---|---|
| `f1_macro_base` | 0.856 |
| `f1_macro_baseline` | 0.0554 |
| `f1_macro_automatica` | 0.9308 |
| `cobertura_ponto_operacao` | 0.7666 |
| `limiar_operacao` | 0.5 |
| `composicao_desvio_sobre` | Hardware, Miscellaneous, HR Support |
| `composicao_desvio_sub` | Administrative rights, Access, Internal Project, Storage, Purchase |

### Síntese do Bloco 4

O protótipo fecha a proposta do Bloco 3 com número real. Um classificador linear
sobre o texto do D2 recupera F1-macro 0.856, mais de quinze vezes o baseline. A
camada de abstenção no limiar 0.5 automatiza 76.7% dos tickets a F1-macro 0.931 e
desvia o resto para revisão humana. O resíduo que a revisão humana herda é o
ticket de `Hardware` ambíguo, o sorvedouro de confusão de 4.1. O `Miscellaneous`,
categoria difusa no vocabulário pelo Bloco 2, é classificado com F1 0.829, o que
separa vocabulário difuso de dificuldade de classificação. A camada de desvio
transforma o erro de classificação em carga de trabalho explícita e endereçável.